### Genetic Algorithm

Proyek ini mengimplementasikan **Algoritma Genetika (Genetic Algorithm)** untuk seleksi fitur optimal pada dataset dunia nyata (*Heart Disease Dataset* dari Kaggle), sekaligus membandingkan performa ketiga metode seleksi: *Roulette Wheel*, *Tournament*, dan *Rank-Based Selection*.

### Studi Kasus & Konsep Dasar:
* **Cerita:** Menggunakan data rekam medis pasien untuk mencari subset (kombinasi) kolom fitur terbaik yang paling akurat dan efisien dalam memprediksi risiko penyakit jantung.
* **Tujuan GA:** Memilih kombinasi fitur medis seminimal mungkin namun tetap menghasilkan nilai *fitness* (akurasi/skor performa) yang setinggi mungkin.
* **Comparative Evaluation:** Menjalankan eksperimen secara *head-to-head* untuk melihat metode seleksi mana di antara *Roulette Wheel*, *Tournament*, dan *Rank-Based* yang paling stabil dan cepat konvergen pada data medis riil.

In [1]:
import random
import pandas as pd
import matplotlib.pyplot as plt

### Load dataset

In [3]:
try:
    df = pd.read_csv('heart.csv')
    print("Berhasil, Jumlah baris:", len(df))
except FileNotFoundError:
    print("Gagal")

Berhasil, Jumlah baris: 1025


### Seleksi Kolom

In [8]:
x = df.iloc[:, :-1]
y = df.iloc[:, -1]
feature_names = list(x.columns)
num_features = len(feature_names)

### Parameter

In [9]:
pop_size = 16
generations = 30
CR = 0.85
MR = 0.1

### Fitness Function

In [12]:
def calculate_fitness(chromosome):
    if sum(chromosome) == 0:
        return 0.01

    selected_features = [feature_names[i] for i in range(num_features) if chromosome[i] == 1]
    subset_df = x[selected_features]
    corellation_sum = subset_df.corrwith(y).abs().mean()

    feature_penalty = 0.01 * len(selected_features)

    fitness = correlation_sum - feature_penalty
    return max(float(fitness), 0.001)

### Populasi awal

In [13]:
def create_chromosome():
    return [random.randint(0, 1) for _ in range(num_features)]

### Metode seleksi (Menggunakan tiga metode yaitu: Roulette Wheel Selection, Tournament Selection, dan Rank Based Selection)

In [16]:
def roulette_wheel_selection(population, fitnesses):
    total_fit = sum(fitnesses)
    if total_fit == 0:
        return random.choice(population).copy()
    pick = random.uniform(0, total_fit)
    current = 0
    for ind, fit in zip(population, fitnesses):
        current += fit
        if current >= pick:
            return ind.copy()
    return population[0].copy()

def tournament_selection(population, fitnesses, k=3):
    selected_indices =  random.sample(range(len(population)), min(k, len(population)))
    best_idx = selected_indices[0]
    for idx in selected_indices:
        if fitnesses[idx] > fitnesses[best_idx]:
            best_idx = idx
    return population[best_idx].copy()

def rank_based_selection(population, fitnesses):
    indexed_fit = sorted(list(enumerate(fitnesses)), key=lambda x: x[1])
    ranks = list(range(1, len(population) + 1))
    total_rank = sum(ranks)

    cumulative, current_sum = [], 0
    for r in ranks:
        current_sum += (r / total_rank)
        cumulative.append(current_sum)

    pick = random.random()
    for i, prob in enumerate(cumulative):
        if pick <= prob:
            orig_idx = indexed_fit[i][0]
            return population[orig_idx].copy()
    return population[indexed_fit[-1][0].copy()]

### Crossover

In [17]:
def crossover(p1, p2):
    if random.random() < CR:
        cut = random.randint(1, num_features -1)
        return p1[:cut] + p2[cut:], p2[:cut] + p1[cut:]
    return p1.copy(), p2.copy()

### Mutation

In [19]:
def mutate(chrom):
    mutated = chrom.copy()
    for i in range(len(mutated)):
        if random.random() < MR:
            mutated[i] = 1 if mutated[i] == 0 else 0
    return mutated

### main loooooop